In [1]:
import os
from pathlib import Path

# Notebook'u bitirme_project klasörü içinde çalıştır (önerilen)
PROJECT_DIR = Path(".").resolve()

# Google service account json (senin bilgisayarında bu dosya olmalı)
KEY_PATH = PROJECT_DIR / "secrets" / "gcp-speech.json"

if not KEY_PATH.exists():
    raise FileNotFoundError(
        f"Credential json bulunamadı: {KEY_PATH}\n"
        f"Dosyayı bu klasöre koy: {PROJECT_DIR / 'secrets'}"
    )

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(KEY_PATH)
print("Credential set:", os.environ["GOOGLE_APPLICATION_CREDENTIALS"])


FileNotFoundError: Credential json bulunamadı: C:\Users\halil melih\BitirmeProject\bitirme_project\src\secrets\gcp-speech.json
Dosyayı bu klasöre koy: C:\Users\halil melih\BitirmeProject\bitirme_project\src\secrets

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(".").resolve()

# Tercih edilen dataset yolu:
AUDIO_DIR = PROJECT_DIR / "data" / "raw" / "911_recordings" / "audio"

# Fallback: eğer tek klasör kullandıysan
FALLBACK_AUDIO_DIR = PROJECT_DIR / "data-raw-911_recordings-audio"

if not AUDIO_DIR.exists() and FALLBACK_AUDIO_DIR.exists():
    AUDIO_DIR = FALLBACK_AUDIO_DIR

print("PROJECT_DIR:", PROJECT_DIR)
print("AUDIO_DIR:", AUDIO_DIR)
print("AUDIO_DIR exists?:", AUDIO_DIR.exists())

mp3_files = sorted(AUDIO_DIR.glob("call_*.mp3"))
print("MP3 count:", len(mp3_files))
mp3_files[:10]


In [ ]:
import os
from google.cloud import speech

creds = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
print("GOOGLE_APPLICATION_CREDENTIALS:", creds)


In [ ]:
import os
import re
import csv
import subprocess
import tempfile
from dataclasses import dataclass
from pathlib import Path

from google.cloud import speech

@dataclass
class TranscriptionResult:
    transcript: str
    confidence: float | None
    language_code: str
    flac_path: str | None = None

def ensure_flac_mono_16k(input_path: str, out_dir: str = ".tmp_audio") -> str:
    # MP3/WAV -> mono + 16kHz + FLAC (ffmpeg gerekir)
    # Zaten .flac ise direkt döner.
    if input_path.lower().endswith(".flac"):
        return input_path

    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(input_path))[0]
    out_path = os.path.join(out_dir, f"{base}.flac")

    cmd = ["ffmpeg", "-y", "-i", input_path, "-ac", "1", "-ar", "16000", "-vn", out_path]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return out_path

def split_to_flac_chunks(input_path: str, chunk_seconds: int = 25) -> list[str]:
    # MP3/WAV -> 16k mono FLAC chunk'lara böler.
    out_dir = Path(tempfile.mkdtemp(prefix="stt_chunks_"))
    out_pattern = str(out_dir / "chunk_%03d.flac")

    cmd = [
        "ffmpeg", "-y", "-i", input_path,
        "-ac", "1", "-ar", "16000",
        "-af", "highpass=f=200,lowpass=f=3000,dynaudnorm",
        "-f", "segment", "-segment_time", str(chunk_seconds),
        out_pattern
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return sorted(str(p) for p in out_dir.glob("chunk_*.flac"))

def build_config(language_code: str = "en-US"):
    # Telefon görüşmeleri için daha iyi ayarlar
    return speech.RecognitionConfig(
        encoding=speech.RecognitionConfig.AudioEncoding.FLAC,
        sample_rate_hertz=16000,
        language_code=language_code,
        enable_automatic_punctuation=True,
        model="phone_call",
        use_enhanced=True,
        audio_channel_count=1,
        max_alternatives=3,
        speech_contexts=[speech.SpeechContext(
            phrases=[
                "Prairie Road", "ditch", "wedding", "field",
                "help me", "i don't know where i am",
                "vehicle", "map", "lost"
            ],
            boost=12.0
        )],
    )

def transcribe_local_file(input_path: str, language_code: str = "en-US") -> TranscriptionResult:
    # Tek bir (küçük) dosyayı Google STT ile çözer
    client = speech.SpeechClient()
    flac_path = ensure_flac_mono_16k(input_path)

    with open(flac_path, "rb") as f:
        content = f.read()

    audio = speech.RecognitionAudio(content=content)
    config = build_config(language_code=language_code)
    response = client.recognize(config=config, audio=audio)

    parts, confs = [], []
    for result in response.results:
        if not result.alternatives:
            continue
        best = max(
            result.alternatives,
            key=lambda a: ((float(a.confidence) if a.confidence else 0.0), len(a.transcript or "")),
        )
        if best.transcript:
            parts.append(best.transcript.strip())
        if best.confidence is not None:
            confs.append(float(best.confidence))

    transcript = " ".join(parts).strip()
    confidence = (sum(confs) / len(confs)) if confs else None
    return TranscriptionResult(transcript, confidence, language_code, flac_path)

def transcribe_local_file_chunked(input_path: str, language_code: str = "en-US", chunk_seconds: int = 25):
    chunks = split_to_flac_chunks(input_path, chunk_seconds=chunk_seconds)
    full_text, confs = [], []

    for i, ch in enumerate(chunks, 1):
        size_mb = os.path.getsize(ch) / (1024 * 1024)
        print(f"[{i}/{len(chunks)}] {Path(ch).name} ({size_mb:.2f} MB)")
        r = transcribe_local_file(ch, language_code=language_code)
        if r.transcript:
            full_text.append(r.transcript.strip())
        if r.confidence is not None:
            confs.append(r.confidence)

    return {
        "transcript": "\n".join(full_text),
        "avg_confidence": (sum(confs)/len(confs)) if confs else None,
        "chunks": len(chunks),
    }

def batch_transcribe_to_csv(
    mp3_files: list[Path],
    out_csv: Path,
    language_code: str = "en-US",
    chunk_seconds: int = 25,
    limit: int | None = None,
):
    # Tüm MP3'leri transkribe edip CSV'ye yazar: path,text,avg_confidence,chunks
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    files = mp3_files[:limit] if limit else mp3_files

    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "text", "avg_confidence", "chunks"])
        w.writeheader()

        for idx, p in enumerate(files, 1):
            print(f"\n=== [{idx}/{len(files)}] {p.name} ===")
            out = transcribe_local_file_chunked(str(p), language_code=language_code, chunk_seconds=chunk_seconds)
            w.writerow({
                "path": p.relative_to(Path('.').resolve()).as_posix(),
                "text": out["transcript"].replace("\r", " ").strip(),
                "avg_confidence": out["avg_confidence"],
                "chunks": out["chunks"],
            })

    print("\nWrote CSV:", out_csv, "rows:", len(files))


In [ ]:
import re
from pathlib import Path

def call_number(p: Path) -> int:
    m = re.search(r"call_(\d+)$", p.stem)
    return int(m.group(1)) if m else 10**9

mp3_files = sorted(AUDIO_DIR.glob("call_*.mp3"), key=call_number)

# Örnek: call_9
sample = next(p for p in mp3_files if call_number(p) == 9)
print("Transcribing (chunked):", sample)

out = transcribe_local_file_chunked(str(sample), language_code="en-US", chunk_seconds=25)
print(out["transcript"][:2000])
print("Chunks:", out["chunks"], "Avg conf:", out["avg_confidence"])

# === Toplu transkripti CSV'ye yazmak için ===
# OUT_CSV = PROJECT_DIR / "data" / "labels" / "auto_transcripts.csv"
# batch_transcribe_to_csv(mp3_files, OUT_CSV, language_code="en-US", chunk_seconds=25, limit=20)  # limit=20 test için
